In [0]:
%sql
select * from workspace.silver.silver_flights

flight_id,airline,origin,destination,flight_date,modifiedDate
F0001,Delta,Kellyfort,South Kathleen,2025-05-04,2025-09-05T13:43:47.644Z
F0002,Qatar Airways,Lake Stephen,New Vincent,2025-04-29,2025-09-05T13:43:47.644Z
F0004,Delta,Maddenshire,Johnchester,2025-05-16,2025-09-05T13:43:47.644Z
F0006,Air Canada,New Richardside,South Jamesborough,2025-05-16,2025-09-05T13:43:47.644Z
F0007,Delta,Berryport,Miguelburgh,2025-05-24,2025-09-05T13:43:47.644Z
F0008,Lufthansa,Briannachester,Cervantesland,2025-05-26,2025-09-05T13:43:47.644Z
F0009,Delta,Alexandraborough,North Alexishaven,2025-06-10,2025-09-05T13:43:47.644Z
F0010,Emirates,Kruegerchester,Martintown,2025-05-20,2025-09-05T13:43:47.644Z
F0011,Lufthansa,Port Joannahaven,North Jasonton,2025-05-30,2025-09-05T13:43:47.644Z
F0012,Air Canada,New Hunter,East Maria,2025-04-23,2025-09-05T13:43:47.644Z


Dimension tables have a primary key called as **dim surrogate key** which is artificially generated.

**Slowly changing dimension**\
A dimension table has 5 records with surrogate keys 1,2,3,4,5. In new load data for record 2,3 is changed and also new data is added. So surrogate key for 2,3 should not change but only data is updated and new record must get key as 6. This is called Slowly changing dimension. By definition A slowly changing dimension (SCD) in data warehousing is a dimensional attribute that changes over time, requiring strategies to manage these changes and preserve historical data for analysis. Types are:

- Type 1 (Update in Place): The old attribute value is simply overwritten with the new value (this project). 
- Type 2 (Preserve History): New records are created to capture the changed attribute, while the old record remains to preserve history. 
- Type 3 (Keep Previous Value): The current and previous values of an attribute are kept in the same dimension record. 

### **Static approach using widgets**

In [0]:
# # key columns
# dbutils.widgets.text('keycols', '')

# # CDC columns
# dbutils.widgets.text('cdccols', '')

# # Back-dated refresh
# dbutils.widgets.text('backdated_refresh', '')

# # Source object
# dbutils.widgets.text('source_object', '')

# # Source schema
# dbutils.widgets.text('source_schema', '')

# # CDC column
# cdc_col = dbutils.widgets.get('cdccols')

# # Key columns list
# key_cols = dbutils.widgets.get('keycols')
# key_cols_list = eval(key_cols)                # need to pass a value in keycols widget else will throw error

# # Back-dated refresh
# backdated_refresh = dbutils.widgets.get('backdated_refresh')

# # Source object
# source_object = dbutils.widgets.get('source_object')

# # Source schema
# source_schema = dbutils.widgets.get('source_schema')

### **Parameters**

In [0]:
# CDC column                        #  column used to track and manage changes to data in a database
cdc_col = 'modifiedDate'

# Key columns list
key_cols = "['passenger_id']"          # Primary key for flights
key_cols_list = eval(key_cols)               

# Back-dated refresh
backdated_refresh = ""

# Source object                     # flights table name in silver
source_object = 'silver_passengers'

# Source schema
source_schema = 'silver'

# Target object                     
target_object = 'DimPassengers'

# Target schema
target_schema = 'gold'

# Surrogate key 
surrogate_key = 'DimPassengersKey'


### **Incremental Data Ingestion**

#### **Last load date**


In [0]:
# The purpose of this code is to determine the last_load timestamp for an incremental load process, based on whether a table already exists or whether a backdated refresh is requested.

# Check if backdated refresh is not provided (empty string)
if backdated_refresh == "":

  # If the target table already exists in the workspace
  if spark.catalog.tableExists(f'workspace.{target_schema}.{target_object}'):

    # Get the maximum modifiedDate (latest change) from the target table
    last_load = spark.sql(f'select max({cdc_col}) from workspace.{target_schema}.{target_object}').collect()[0][0]
  
  # If the target table does not exist
  else : 
    # Set last_load to a very old default date (acts as a starting point)
    last_load = '1900-01-01 00:00:00'

# If backdated refresh is provided
else:
  # Use the given backdated refresh timestamp as last_load
  last_load = backdated_refresh

last_load  

datetime.datetime(2025, 9, 5, 13, 43, 47, 636000)

In [0]:
df_src = spark.sql(f'select * from workspace.{source_schema}.{source_object} where {cdc_col} >= "{last_load}"')

df_src.display()

passenger_id,name,gender,nationality,modifiedDate
P0001,Kevin Ferguson,Male,Reunion,2025-09-05T13:43:47.636Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2025-09-05T13:43:47.636Z
P0003,Cynthia Frazier,Male,Marshall Islands,2025-09-05T13:43:47.636Z
P0004,Ryan Ramsey,Male,Niger,2025-09-05T13:43:47.636Z
P0005,Mike Kim,Male,Taiwan,2025-09-05T13:43:47.636Z
P0006,Diana Adams,Male,Mayotte,2025-09-05T13:43:47.636Z
P0007,Sharon Moon,Male,Madagascar,2025-09-05T13:43:47.636Z
P0008,Cheryl Glenn,Male,Maldives,2025-09-05T13:43:47.636Z
P0009,Allen Lowery,Male,Rwanda,2025-09-05T13:43:47.636Z
P0010,Maria Medina,Male,Denmark,2025-09-05T13:43:47.636Z


### Old vs New records

This Spark code is handling a scenario where it checks whether a target table exists and then builds a DataFrame (df_target) either by selecting from that table (if it exists) or by creating an empty DataFrame with the same schema (if it doesn’t exist).


In [0]:
# Check if the target table exists in the given schema
if spark.catalog.tableExists(f'workspace.{target_schema}.{target_object}'):

    # Join key columns into a comma-separated string for SQL query (incremental load case)
    key_cols_string_incremental = ', '.join(key_cols_list)

    # Select key columns, surrogate key, create_date, and update_date from the existing target table
    df_target = spark.sql(f' select {key_cols_string_incremental}, {surrogate_key}, create_date, update_date from workspace.{target_schema}.{target_object}')

# if target table doesnt exist
else:

    # Build placeholder expressions ('' as column_name) for key columns (initial load case)
    key_cols_string_init = [f"'' as {i}" for i in key_cols_list]
    # Join placeholder expressions into a comma-separated string
    key_cols_string_init = ', '.join(key_cols_string_init)

    # Create an empty DataFrame with the same schema 
    # where condition is applied to get no records
    df_target = spark.sql(f"select {key_cols_string_init}, cast('0' as int) as {surrogate_key}, cast('1900-01-01 00:00:00' as timestamp) as create_date, cast('1900-01-01 00:00:00' as timestamp) as update_date where 1=0")

df_target.display()

passenger_id,DimPassengersKey,create_date,update_date
P0001,1,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0002,2,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0003,3,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0004,4,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0005,5,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0006,6,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0007,7,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0008,8,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0009,9,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0010,10,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z


**Join condition**

In [0]:
join_condition = ' and '.join([f"src.{i} = trg.{i}" for i in key_cols_list])

In [0]:
df_src.createOrReplaceTempView("src")
df_target.createOrReplaceTempView("trg")

df_join = spark.sql(f"""
            select  src.*,
                    trg.{surrogate_key}, trg.create_date, trg.update_date
            from src
            left join trg
            on {join_condition} """)

In [0]:
from pyspark.sql.functions import *

# Old records
df_old = df_join.filter(col(f'{surrogate_key}').isNotNull())

# New records
df_new = df_join.filter(col(f'{surrogate_key}').isNull())

df_old.display()


passenger_id,name,gender,nationality,modifiedDate,DimPassengersKey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2025-09-05T13:43:47.636Z,1,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2025-09-05T13:43:47.636Z,2,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0003,Cynthia Frazier,Male,Marshall Islands,2025-09-05T13:43:47.636Z,3,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0004,Ryan Ramsey,Male,Niger,2025-09-05T13:43:47.636Z,4,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0005,Mike Kim,Male,Taiwan,2025-09-05T13:43:47.636Z,5,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0006,Diana Adams,Male,Mayotte,2025-09-05T13:43:47.636Z,6,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0007,Sharon Moon,Male,Madagascar,2025-09-05T13:43:47.636Z,7,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0008,Cheryl Glenn,Male,Maldives,2025-09-05T13:43:47.636Z,8,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0009,Allen Lowery,Male,Rwanda,2025-09-05T13:43:47.636Z,9,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z
P0010,Maria Medina,Male,Denmark,2025-09-05T13:43:47.636Z,10,2025-09-09T03:18:16.223Z,2025-09-09T03:18:16.223Z


### **Enriching dataframes**

#### **Preparing df_old_enriched**

In [0]:
df_old_enriched = df_old.withColumn('update_date', current_timestamp())

df_old_enriched.display()

passenger_id,name,gender,nationality,modifiedDate,DimPassengersKey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2025-09-05T13:43:47.636Z,1,2025-09-09T03:18:16.223Z,2025-09-10T01:50:58.725Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2025-09-05T13:43:47.636Z,2,2025-09-09T03:18:16.223Z,2025-09-10T01:50:58.725Z
P0003,Cynthia Frazier,Male,Marshall Islands,2025-09-05T13:43:47.636Z,3,2025-09-09T03:18:16.223Z,2025-09-10T01:50:58.725Z
P0004,Ryan Ramsey,Male,Niger,2025-09-05T13:43:47.636Z,4,2025-09-09T03:18:16.223Z,2025-09-10T01:50:58.725Z
P0005,Mike Kim,Male,Taiwan,2025-09-05T13:43:47.636Z,5,2025-09-09T03:18:16.223Z,2025-09-10T01:50:58.725Z
P0006,Diana Adams,Male,Mayotte,2025-09-05T13:43:47.636Z,6,2025-09-09T03:18:16.223Z,2025-09-10T01:50:58.725Z
P0007,Sharon Moon,Male,Madagascar,2025-09-05T13:43:47.636Z,7,2025-09-09T03:18:16.223Z,2025-09-10T01:50:58.725Z
P0008,Cheryl Glenn,Male,Maldives,2025-09-05T13:43:47.636Z,8,2025-09-09T03:18:16.223Z,2025-09-10T01:50:58.725Z
P0009,Allen Lowery,Male,Rwanda,2025-09-05T13:43:47.636Z,9,2025-09-09T03:18:16.223Z,2025-09-10T01:50:58.725Z
P0010,Maria Medina,Male,Denmark,2025-09-05T13:43:47.636Z,10,2025-09-09T03:18:16.223Z,2025-09-10T01:50:58.725Z


#### **Preparing df_new_enriched**

This block is handling surrogate key generation and enrichment of a new DataFrame depending on whether the target table exists or not.

In [0]:
# Check if the target table exists in the given schema
if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):

    # Get the current maximum surrogate key from the existing target table
    max_surrogate_key = spark.sql(f"""
                                  select max({surrogate_key}) from workspace.{target_schema}.{target_object}
                                  """).collect()[0][0]
    
    # If target table exists, start surrogate keys from the next available number (incremental load case)
    df_new_enriched = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key + 1 + monotonically_increasing_id())).withColumn('create_date', current_timestamp()).withColumn('update_date', current_timestamp())

else:
    # If target table doesn’t exist, start surrogate keys from 0 (initial load case)
    max_surrogate_key = 0
    df_new_enriched = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key + 1 + monotonically_increasing_id())).withColumn('create_date', current_timestamp()).withColumn('update_date', current_timestamp())
    
df_new_enriched.display()    

passenger_id,name,gender,nationality,modifiedDate,DimPassengersKey,create_date,update_date
P0225,William Lopez,Male,Heard Island and McDonald Islands,2025-09-10T01:39:42.890Z,221,2025-09-10T01:51:00.322Z,2025-09-10T01:51:00.322Z
P0224,Jason Jensen,Female,Rwanda,2025-09-10T01:39:42.890Z,222,2025-09-10T01:51:00.322Z,2025-09-10T01:51:00.322Z
P0222,Maria Taylor,Male,Lao People's Democratic Republic,2025-09-10T01:39:42.890Z,223,2025-09-10T01:51:00.322Z,2025-09-10T01:51:00.322Z
P0223,Nicholas Gomez,Female,Cook Islands,2025-09-10T01:39:42.890Z,224,2025-09-10T01:51:00.322Z,2025-09-10T01:51:00.322Z
P0221,Amy Welch,Male,Croatia,2025-09-10T01:39:42.890Z,225,2025-09-10T01:51:00.322Z,2025-09-10T01:51:00.322Z


In [0]:
max_surrogate_key

220

In [0]:
df_old_enriched.display()

passenger_id,name,gender,nationality,modifiedDate,DimPassengersKey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2025-09-05T13:43:47.636Z,1,2025-09-09T03:18:16.223Z,2025-09-10T01:51:01.604Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2025-09-05T13:43:47.636Z,2,2025-09-09T03:18:16.223Z,2025-09-10T01:51:01.604Z
P0003,Cynthia Frazier,Male,Marshall Islands,2025-09-05T13:43:47.636Z,3,2025-09-09T03:18:16.223Z,2025-09-10T01:51:01.604Z
P0004,Ryan Ramsey,Male,Niger,2025-09-05T13:43:47.636Z,4,2025-09-09T03:18:16.223Z,2025-09-10T01:51:01.604Z
P0005,Mike Kim,Male,Taiwan,2025-09-05T13:43:47.636Z,5,2025-09-09T03:18:16.223Z,2025-09-10T01:51:01.604Z
P0006,Diana Adams,Male,Mayotte,2025-09-05T13:43:47.636Z,6,2025-09-09T03:18:16.223Z,2025-09-10T01:51:01.604Z
P0007,Sharon Moon,Male,Madagascar,2025-09-05T13:43:47.636Z,7,2025-09-09T03:18:16.223Z,2025-09-10T01:51:01.604Z
P0008,Cheryl Glenn,Male,Maldives,2025-09-05T13:43:47.636Z,8,2025-09-09T03:18:16.223Z,2025-09-10T01:51:01.604Z
P0009,Allen Lowery,Male,Rwanda,2025-09-05T13:43:47.636Z,9,2025-09-09T03:18:16.223Z,2025-09-10T01:51:01.604Z
P0010,Maria Medina,Male,Denmark,2025-09-05T13:43:47.636Z,10,2025-09-09T03:18:16.223Z,2025-09-10T01:51:01.604Z


#### **Unioning old and new records**

In [0]:
# unionbyname appends the table wrt the schema of the first table
df_union = df_old_enriched.unionByName(df_new_enriched)
df_union.display()

passenger_id,name,gender,nationality,modifiedDate,DimPassengersKey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2025-09-05T13:43:47.636Z,1,2025-09-09T03:18:16.223Z,2025-09-10T01:51:02.650Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2025-09-05T13:43:47.636Z,2,2025-09-09T03:18:16.223Z,2025-09-10T01:51:02.650Z
P0003,Cynthia Frazier,Male,Marshall Islands,2025-09-05T13:43:47.636Z,3,2025-09-09T03:18:16.223Z,2025-09-10T01:51:02.650Z
P0004,Ryan Ramsey,Male,Niger,2025-09-05T13:43:47.636Z,4,2025-09-09T03:18:16.223Z,2025-09-10T01:51:02.650Z
P0005,Mike Kim,Male,Taiwan,2025-09-05T13:43:47.636Z,5,2025-09-09T03:18:16.223Z,2025-09-10T01:51:02.650Z
P0006,Diana Adams,Male,Mayotte,2025-09-05T13:43:47.636Z,6,2025-09-09T03:18:16.223Z,2025-09-10T01:51:02.650Z
P0007,Sharon Moon,Male,Madagascar,2025-09-05T13:43:47.636Z,7,2025-09-09T03:18:16.223Z,2025-09-10T01:51:02.650Z
P0008,Cheryl Glenn,Male,Maldives,2025-09-05T13:43:47.636Z,8,2025-09-09T03:18:16.223Z,2025-09-10T01:51:02.650Z
P0009,Allen Lowery,Male,Rwanda,2025-09-05T13:43:47.636Z,9,2025-09-09T03:18:16.223Z,2025-09-10T01:51:02.650Z
P0010,Maria Medina,Male,Denmark,2025-09-05T13:43:47.636Z,10,2025-09-09T03:18:16.223Z,2025-09-10T01:51:02.650Z


### UPSERT (update + insert)
This code checks if a Delta table exists in Spark, and then either merges new data into it (upsert logic) or creates it if it doesn’t exist.

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):
    
    # Get a reference to the existing Delta table
    dlt_obj = DeltaTable.forName(spark, f"workspace.{target_schema}.{target_object}")

    # Start a merge (upsert) between source dataframe and target Delta table using surrogate key
    ( dlt_obj.alias("trg").merge(df_union.alias("src"), f"trg.{surrogate_key} = src.{surrogate_key}")\
            # Update if a match is found and only if source record is newer (based on CDC column)
            .whenMatchedUpdateAll(condition = f"src.{cdc_col} >= trg.{cdc_col}")\
            # Insert the row if it doesn’t exist in the target    
            .whenNotMatchedInsertAll()\
            .execute() )

 # If the target Delta table doesn’t exist
else:

    ( df_union.write.format("delta")\
        # Append mode (creates the table if it doesn’t exist)
        .mode("append")\
        # Save the dataframe as a new managed Delta table as gold layer in the target schema
        .saveAsTable(f"workspace.{target_schema}.{target_object}") )               

In [0]:
spark.sql(f"select * from workspace.{target_schema}.{target_object}").display()

passenger_id,name,gender,nationality,modifiedDate,DimPassengersKey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2025-09-05T13:43:47.636Z,1,2025-09-09T03:18:16.223Z,2025-09-10T01:51:04.386Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2025-09-05T13:43:47.636Z,2,2025-09-09T03:18:16.223Z,2025-09-10T01:51:04.386Z
P0003,Cynthia Frazier,Male,Marshall Islands,2025-09-05T13:43:47.636Z,3,2025-09-09T03:18:16.223Z,2025-09-10T01:51:04.386Z
P0004,Ryan Ramsey,Male,Niger,2025-09-05T13:43:47.636Z,4,2025-09-09T03:18:16.223Z,2025-09-10T01:51:04.386Z
P0005,Mike Kim,Male,Taiwan,2025-09-05T13:43:47.636Z,5,2025-09-09T03:18:16.223Z,2025-09-10T01:51:04.386Z
P0006,Diana Adams,Male,Mayotte,2025-09-05T13:43:47.636Z,6,2025-09-09T03:18:16.223Z,2025-09-10T01:51:04.386Z
P0007,Sharon Moon,Male,Madagascar,2025-09-05T13:43:47.636Z,7,2025-09-09T03:18:16.223Z,2025-09-10T01:51:04.386Z
P0008,Cheryl Glenn,Male,Maldives,2025-09-05T13:43:47.636Z,8,2025-09-09T03:18:16.223Z,2025-09-10T01:51:04.386Z
P0009,Allen Lowery,Male,Rwanda,2025-09-05T13:43:47.636Z,9,2025-09-09T03:18:16.223Z,2025-09-10T01:51:04.386Z
P0010,Maria Medina,Male,Denmark,2025-09-05T13:43:47.636Z,10,2025-09-09T03:18:16.223Z,2025-09-10T01:51:04.386Z


Change the parameters for each dimension and run the notebook again. The data will be stored as delta table in the gold schema

You can upload the SCD file in the raw volume and run the bronze job to see the incremental load and then run the silver pipeline to see the SCD logic and then run the gold notebook to see the final table with surrogate key and updated data with date